# 02 - Regresión tabular profunda en California Housing

**Comparación reproducible de Ridge, TabNet, TabTransformer, FT-Transformer y SAINT supervisado**

Este cuaderno estudia un problema de regresión tabular con una base experimental común. El propósito no es proclamar una arquitectura ganadora de antemano, sino observar qué representaciones y sesgos inductivos resultan útiles en un dataset pequeño, completamente numérico y con estructura geográfica.

## 1. Contexto y objetivos

California Housing contiene estadísticas agregadas de bloques censales de California. El objetivo `MedHouseVal` representa el valor mediano de vivienda en unidades de 100.000 dólares estadounidenses. La comparación mantendrá constantes las particiones, el preprocesamiento, las semillas y el protocolo de evaluación.

Los objetivos son construir un baseline lineal exigente pero sencillo, comparar cuatro familias de deep learning tabular, medir error y costo computacional, y detectar limitaciones que puedan invalidar una interpretación demasiado optimista.

## 2. Preguntas experimentales

La evaluación se organiza alrededor de preguntas que solo se responderán después de observar los resultados:

1. ¿Algún modelo profundo reduce el RMSE y el MAE frente a Ridge?
2. ¿Las mejoras, si aparecen, compensan el costo de entrenamiento y el número de parámetros?
3. ¿Qué patrones sistemáticos permanecen en los residuos?
4. ¿Cómo afecta la naturaleza completamente numérica del dataset a cada arquitectura?
5. ¿Son estables las conclusiones al repetir el experimento con varias semillas?

## 3. Importaciones y dependencias

El cuaderno se limita a configuración, ejecución y presentación. La carga y transformación de datos, las arquitecturas, el entrenamiento y la evaluación permanecen en `src/`. Esta separación reduce duplicación y permite verificar cada responsabilidad de forma independiente.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.data import (
    RegressionSplitConfig,
    audit_regression_dataframe,
    load_regression_dataframe,
    make_california_housing_config,
    prepare_regression_data,
    sample_regression_dataframe,
)
from src.evaluation import (
    plot_regression_costs,
    plot_regression_histories,
    plot_regression_predictions,
    plot_regression_residuals,
    regression_comparison_table,
    validate_regression_experiment_results,
)
from src.models import optional_dependency_report
from src.training import (
    finalize_regression_experiment,
    resolve_device,
    run_regression_experiment,
    set_reproducible_seed,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
pd.set_option("display.max_columns", 120)

DEPENDENCY_REPORT = optional_dependency_report()
missing_dependencies = [
    name for name, available in DEPENDENCY_REPORT.items() if not available
]
if missing_dependencies:
    raise ImportError(f"Missing required dependencies: {missing_dependencies}")

DEPENDENCY_VERSIONS = {
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "scikit-learn": version("scikit-learn"),
    "torch": version("torch"),
    "pytorch-tabnet": version("pytorch-tabnet"),
}
DEPENDENCY_VERSIONS

## 4. Configuración central

Todos los valores que determinan el experimento se declaran una sola vez. Las configuraciones específicas respetan el diseño de cada arquitectura; igualdad de protocolo no significa imponer idénticas representaciones internas. Se usan tres semillas sobre un split fijo y `device="auto"`, que selecciona CUDA cuando está disponible.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Run this notebook from the repository root."
    )

DATA_DIR = PROJECT_ROOT / "archive"
RESULTS_DIR = PROJECT_ROOT / "results" / "regression"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

EXPERIMENT_CONFIG = {
    "experiment_id": "california_housing_dl_architectures_v2",
    "seeds": [42, 123, 2025],
    "seed": 42,
    "device": "auto",
    "deterministic": True,
    "download_if_missing": True,
    "sample_size": None,
    "batch_size": 512,
    "inference_batch_size": 2048,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "max_epochs": 60,
    "patience": 10,
    "num_workers": 0,
    "selection_metric": "rmse",
    "defer_test_until_final": True,
    "require_row_independent_inference": True,
    "dependency_versions": DEPENDENCY_VERSIONS,
    "max_grad_norm": 5.0,
    "results_dir": RESULTS_DIR,
    "model_configs": {
        "baseline_ridge": {
            "alpha": 1.0,
            "solver": "auto",
        },
        "tabnet": {
            "n_d": 16,
            "n_a": 16,
            "n_steps": 4,
            "gamma": 1.4,
            "lambda_sparse": 1e-4,
            "batch_size": 512,
            "virtual_batch_size": 128,
        },
        "tab_transformer": {
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
            "mlp_hidden": (128, 64),
        },
        "ft_transformer": {
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
        },
        "saint_supervised": {
            "implementation_version": "saint_column_v1",
            "d_token": 32,
            "n_heads": 4,
            "n_layers": 2,
            "dropout": 0.10,
            "mlp_hidden": (128, 64),
            "numerical_embedding_hidden": 32,
            "use_row_attention": False,
        },
    },
}

DATASET_CONFIG = make_california_housing_config(DATA_DIR)
SPLIT_CONFIG = RegressionSplitConfig(
    train_size=0.70,
    valid_size=0.15,
    test_size=0.15,
    random_state=EXPERIMENT_CONFIG["seed"],
    strategy="random",
)
MODEL_ORDER = [
    "baseline_ridge",
    "tabnet",
    "tab_transformer",
    "ft_transformer",
    "saint_supervised",
]

## 5. Semillas y dispositivo

Python, NumPy y PyTorch reciben la misma semilla. En CUDA se desactivan los kernels Flash y memory-efficient de atención y se exigen algoritmos deterministas; una operación incompatible genera un error en lugar de continuar silenciosamente. Versiones de bibliotecas y dispositivos distintos todavía pueden producir pequeñas diferencias numéricas, por lo que reproducibilidad no equivale a identidad universal entre máquinas.

In [ ]:
set_reproducible_seed(
    EXPERIMENT_CONFIG["seed"],
    deterministic=EXPERIMENT_CONFIG["deterministic"],
)
DEVICE = resolve_device(EXPERIMENT_CONFIG["device"])
EXPERIMENT_CONFIG["device"] = DEVICE

print(f"Device resolved for this run: {DEVICE}")
print("Optional dependency report:", optional_dependency_report())

## 6. Carga de California Housing

`fetch_california_housing` obtiene la copia mantenida por scikit-learn y la almacena en `archive/`. Una vez presente la caché, el entrenamiento no necesita conectividad. La función de carga valida el nombre del target y el esquema esperado antes de devolver el `DataFrame`.

In [ ]:
raw_df = load_regression_dataframe(
    DATASET_CONFIG,
    download_if_missing=EXPERIMENT_CONFIG["download_if_missing"],
)
print(f"Dataset shape: {raw_df.shape}")
display(raw_df.head())

## 7. Auditoría estructural y de calidad

La auditoría comprueba tipos, valores ausentes o infinitos, duplicados y estadísticos del objetivo. También cuantifica cuántas observaciones coinciden con el máximo registrado; esa acumulación es relevante porque el target original está limitado en su extremo superior y los errores cerca del techo no se interpretan como en una variable continua sin censura.

In [ ]:
audit = audit_regression_dataframe(raw_df, DATASET_CONFIG)
print("Shape:", audit["shape"])
print("Missing values:", audit["missing_values"])
print("Infinite values:", audit["infinite_values"])
print("Duplicate rows:", audit["duplicate_rows"])
print("Rows at observed target maximum:", audit["rows_at_observed_target_maximum"])
display(pd.Series(audit["target_summary"], name="target_summary"))

## 8. Definición del target y las características

Los ocho predictores son numéricos. `Latitude` y `Longitude` son características legítimas, pero revelan proximidad espacial; no son fuga de información por sí mismas. La fuga aparecería si el preprocesamiento utilizara filas de validación o prueba, o si bloques vecinos casi idénticos quedaran a ambos lados de una partición que pretende medir generalización geográfica.

In [ ]:
print("Target:", DATASET_CONFIG.target_name)
print("Target unit:", DATASET_CONFIG.target_unit)
print("Numerical columns:", DATASET_CONFIG.numerical_columns)
print("Categorical columns:", DATASET_CONFIG.categorical_columns)
print("Excluded columns:", DATASET_CONFIG.excluded_columns)
print("Identifier columns:", DATASET_CONFIG.identifier_columns)
print("Leakage columns:", DATASET_CONFIG.leakage_columns)
print("Spatial columns:", DATASET_CONFIG.spatial_columns)

## 9. Distribución del objetivo

RMSE da más peso a errores grandes, MAE mide desviación absoluta media y la mediana del error absoluto es resistente a unos pocos fallos extremos. R² expresa la fracción de variabilidad explicada respecto a predecir la media. MAPE se reportará como complemento, pero puede amplificarse en viviendas de bajo valor y no debe ser el único criterio.

In [ ]:
target = raw_df[DATASET_CONFIG.target_name]
display(target.describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]))

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig, axis = plt.subplots(figsize=(8, 4.5))
axis.hist(target, bins=60, color="#287271", alpha=0.85)
axis.axvline(target.max(), color="#c44e52", linestyle="--", label="observed maximum")
axis.set_xlabel("MedHouseVal (100,000 USD)")
axis.set_ylabel("Rows")
axis.set_title("California Housing target distribution")
axis.legend()
axis.grid(axis="y", alpha=0.2)
target_distribution_path = FIGURES_DIR / "regression_target_distribution.png"
fig.tight_layout()
fig.savefig(target_distribution_path, dpi=160)
plt.show()
target_distribution_path

## 10. Partición train/validation/test

Se reserva 70 % para ajustar parámetros, 15 % para early stopping y 15 % para una evaluación final. Validación y prueba cumplen funciones distintas: reutilizar test para escoger épocas o configuraciones convertiría su error en una estimación optimista. Todos los modelos reciben exactamente los mismos índices.

Esta primera exploración usa una partición aleatoria reproducible. No mide generalización a regiones geográficas nuevas; esa limitación se examina al final.

In [ ]:
working_df = sample_regression_dataframe(
    raw_df,
    sample_size=EXPERIMENT_CONFIG["sample_size"],
    random_state=EXPERIMENT_CONFIG["seed"],
)
prepared_data = prepare_regression_data(
    working_df,
    DATASET_CONFIG,
    SPLIT_CONFIG,
)
print(prepared_data.split_report["split_sizes"])
print("Split fingerprint:", prepared_data.split_fingerprint())

## 11. Preprocesamiento ajustado únicamente con train

Cada variable se imputa con la mediana de entrenamiento y se estandariza con media y desviación calculadas en entrenamiento. El objetivo también se estandariza para optimizar las redes con una escala numérica estable; todas las predicciones se devuelven después a unidades originales antes de medir errores. Validación y test nunca reajustan estos parámetros.

In [ ]:
state = prepared_data.preprocessing_state
feature_state = state.feature_state

print("Feature shapes:")
print("  train:", prepared_data.X_train.shape)
print("  valid:", prepared_data.X_valid.shape)
print("  test :", prepared_data.X_test.shape)
print("Categorical matrix shape:", prepared_data.X_cat_train.shape)
print("Preprocessing fitted on:", state.fitted_on)
print("Fit row count:", state.fit_row_count)
print("Target mean fitted on train:", state.target_mean)
print("Target std fitted on train:", state.target_std)
display(
    pd.DataFrame(
        {
            "median": feature_state.numerical_medians,
            "mean": feature_state.numerical_means,
            "std": feature_state.numerical_stds,
        }
    )
)

## 12. Verificaciones contra leakage y errores de forma

Las comprobaciones siguientes son aserciones ejecutables, no mensajes decorativos. Verifican cobertura y separación de índices, ausencia de valores no finitos, coherencia del escalado del target, ajuste exclusivo con train y reproducibilidad exacta de la partición con la misma semilla.

In [ ]:
checks = prepared_data.split_report
reprepared_data = prepare_regression_data(
    working_df,
    DATASET_CONFIG,
    SPLIT_CONFIG,
)

assert checks["no_index_overlap"]
assert checks["all_rows_assigned_once"]
assert checks["no_nan_after_preprocessing"]
assert checks["targets_are_finite"]
assert checks["target_scaling_is_consistent"]
assert checks["cardinalities_are_valid"]
assert checks["preprocessing_fitted_only_on_train"]
assert checks["test_split_used_for_model_selection"] is False
assert checks["target_distribution_reasonable"]
assert np.array_equal(prepared_data.train_indices, reprepared_data.train_indices)
assert np.array_equal(prepared_data.valid_indices, reprepared_data.valid_indices)
assert np.array_equal(prepared_data.test_indices, reprepared_data.test_indices)
assert prepared_data.split_fingerprint() == reprepared_data.split_fingerprint()
np.testing.assert_allclose(prepared_data.y_train_scaled.mean(), 0.0, atol=1e-6)
np.testing.assert_allclose(prepared_data.y_train_scaled.std(), 1.0, atol=1e-6)

checks["same_seed_split_reproducibility"] = True
display(pd.Series(checks, name="integrity_check"))

## 13. Baseline clásico: Ridge

**Principio arquitectónico.** Ridge es una regresión lineal con penalización L2. Recibe los predictores numéricos estandarizados y estima una combinación aditiva.

**Sesgo inductivo.** Favorece relaciones globales lineales y coeficientes de magnitud moderada. Es rápido, estable y difícil de superar por accidente cuando la señal principal es aproximadamente aditiva.

**Ventajas y límites.** Proporciona una referencia interpretable y barata, pero no representa interacciones no lineales ni cambios locales de régimen. No es un encoder latente: su papel es demostrar si la complejidad adicional aporta una reducción real del error.

In [ ]:
experiment_results = []
results_by_model = {}


def validation_table(model_results: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "model_name": result["model_name"],
                "seed": result["seed"],
                **result["valid_metrics"],
            }
            for result in model_results
        ]
    )


def execute_model(model_name: str) -> pd.DataFrame:
    model_results = []
    for seed in EXPERIMENT_CONFIG["seeds"]:
        run_config = dict(EXPERIMENT_CONFIG)
        run_config["seed"] = seed
        model_result = run_regression_experiment(
            model_name=model_name,
            data=prepared_data,
            config=run_config,
        )
        model_results.append(model_result)
    results_by_model[model_name] = model_results
    experiment_results.extend(model_results)
    return validation_table(model_results)


baseline_validation = execute_model("baseline_ridge")
display(baseline_validation)

La tabla anterior corresponde exclusivamente a validación. Ridge no necesita early stopping porque su solución se obtiene directamente; aun así, usa el mismo train y se evalúa sobre la misma validación que las redes. Su resultado fija la referencia que deberá superar cualquier modelo más costoso.

## 14. TabNetRegressor

**Principio arquitectónico.** TabNet aplica una secuencia de pasos de decisión. En cada paso aprende una máscara dispersa sobre columnas y acumula una representación útil para la salida.

**Procesamiento e inductive bias.** En este dataset todas las columnas entran como valores continuos. Las máscaras inducen selección condicional de variables por observación, un sesgo parecido a decisiones por etapas pero entrenado mediante gradientes.

**Ventajas y límites.** Puede capturar interacciones y ofrecer máscaras de importancia, aunque su optimización es sensible al tamaño de red, regularización dispersa y número de pasos. Las máscaras no deben confundirse automáticamente con explicaciones causales.

In [ ]:
tabnet_validation = execute_model("tabnet")
display(tabnet_validation)

La época elegida por TabNet depende solo de su RMSE de validación. El checkpoint restaurado se compara numéricamente con el modelo en memoria antes de persistir resultados; una diferencia por encima de la tolerancia detendría el cuaderno.

## 15. TabTransformer para regresión

**Principio arquitectónico.** TabTransformer contextualiza embeddings de variables categóricas mediante self-attention y concatena después las variables numéricas para la predicción.

**Consecuencia en California Housing.** No existen columnas categóricas. Por ello no hay tokens categóricos ni atención activa en esta implementación; la rama numérica y su MLP constituyen el predictor efectivo. Discretizar variables continuas solo para activar atención cambiaría el problema y añadiría decisiones de binning difíciles de justificar.

**Ventajas y límites.** La ejecución sigue siendo informativa: muestra qué ocurre cuando el sesgo original de una arquitectura no coincide con el esquema del dataset. Sus resultados deben describirse como una variante degenerada numérica, no como evidencia general sobre TabTransformer.

In [ ]:
tab_transformer_validation = execute_model("tab_transformer")
display(tab_transformer_validation)

La interpretación correcta separa implementación y nombre de familia: aquí el bloque Transformer categórico está vacío por construcción. El conteo de parámetros y el historial permiten comprobar qué parte numérica sí fue entrenada.

## 16. FT-Transformer para regresión

**Principio arquitectónico.** FT-Transformer convierte cada columna numérica en un token mediante una transformación afín aprendida, añade un token de agregación y aplica self-attention entre todas las características.

**Sesgo inductivo.** Cada variable conserva identidad propia y puede interactuar globalmente con las demás desde la primera capa. Esto encaja de forma más natural con un dataset totalmente numérico que la contextualización exclusiva de categorías.

**Ventajas y límites.** La atención representa interacciones de alto orden sin ingeniería manual, pero aumenta costo y puede sobreajustar en datasets modestos. El token de agregación produce una representación compacta accesible mediante `get_embedding`.

In [ ]:
ft_transformer_validation = execute_model("ft_transformer")
display(ft_transformer_validation)

El early stopping conserva la época con menor RMSE de validación en unidades originales, aunque la función de pérdida se optimiza sobre el target estandarizado. Esta separación mantiene estabilidad numérica sin alterar la interpretación de la métrica.

## 17. SAINT supervisado para regresión

**Principio arquitectónico.** SAINT crea un token por columna y su formulación completa permite alternar atención entre columnas y entre observaciones. Utiliza un token de agregación para predecir el target.

**Procesamiento e inductive bias.** Cada variable numérica pasa por un embedding no lineal. En esta comparación se usa `saint_column_v1`: la atención entre columnas modela dependencias dentro de una fila y `use_row_attention=False` garantiza inferencia inductiva.

**Ventajas y límites.** La atención intersample completa puede aportar contexto, pero hace que una predicción dependa de la composición del lote. No se activa en el ranking principal porque los demás modelos predicen cada fila de manera independiente. El modelo mantiene la tokenización y atención entre columnas de SAINT y se entrena únicamente con pérdida supervisada.

In [ ]:
saint_validation = execute_model("saint_supervised")
display(saint_validation)

La variante evaluada es invariante al orden y agrupamiento de test. La implementación con atención entre filas permanece disponible como experimento transductivo separado, y el orquestador impide activarla accidentalmente cuando `require_row_independent_inference=True`.

## 18. Evaluación final sobre test

Los hiperparámetros y checkpoints ya quedaron determinados mediante train y validación. Solo ahora se abre la comparación de test. La auditoría exige targets e índices idénticos, el mismo fingerprint, predicciones finitas, métricas recomputables y checkpoints recargados para todas las ejecuciones.

In [ ]:
experiment_results = [
    finalize_regression_experiment(result, prepared_data)
    for result in experiment_results
]
integrity_report = validate_regression_experiment_results(experiment_results)
display(pd.Series(integrity_report, name="final_integrity_check"))

all_results = regression_comparison_table(experiment_results)
display(all_results.sort_values("rmse"))

PLOT_SEED = EXPERIMENT_CONFIG["seeds"][0]
plot_results = [
    result for result in experiment_results if result["seed"] == PLOT_SEED
]

## 19. Tabla comparativa

Ninguna métrica resume por sí sola todo el comportamiento. RMSE enfatiza errores grandes, MAE representa error absoluto promedio, la mediana revela el caso típico, R² mide ajuste relativo y el error medio firmado detecta sesgo global. La media y desviación estándar sobre tres semillas permiten distinguir una diferencia consistente de una fluctuación de inicialización.

In [ ]:
metric_columns = [
    "rmse",
    "mae",
    "median_absolute_error",
    "r2",
    "mape_percent",
    "mean_error",
    "train_time_seconds",
    "inference_time_seconds",
    "n_parameters",
]
summary = all_results.groupby(
    ["model_name", "implementation_version"]
)[metric_columns].agg(["mean", "std"])
display(summary)

model_means = all_results.groupby("model_name")[metric_columns].mean()
ridge_rmse = float(model_means.loc["baseline_ridge", "rmse"])
model_means["rmse_minus_ridge"] = model_means["rmse"] - ridge_rmse
display(model_means.sort_values("rmse"))

## 20. Valores observados frente a predicciones

Una nube cercana a la diagonal indica buen ajuste global. Curvatura, saturación horizontal o compresión de extremos revelan errores estructurados que una métrica agregada puede ocultar. Los paneles usan la primera semilla declarada y comparten límites; la variabilidad entre semillas se resume en la tabla.

In [ ]:
prediction_figure = plot_regression_predictions(
    plot_results,
    FIGURES_DIR,
)
prediction_figure

## 21. Diagnóstico de residuos

Los residuos se definen como observado menos predicho. Una distribución centrada cerca de cero es deseable, pero no suficiente: el gráfico contra la predicción debe mostrar dispersión sin forma sistemática. Abanicos, bandas o asimetrías señalan heterocedasticidad, censura o regiones donde el modelo falla de manera consistente.

In [ ]:
residual_figure = plot_regression_residuals(
    plot_results,
    FIGURES_DIR,
)
residual_figure

## 22. Trayectorias de validación

Las curvas muestran RMSE de validación en unidades originales para la semilla representativa y marcan la mejor época. Una curva todavía descendente al agotar el presupuesto sugiere subentrenamiento; una brecha creciente entre optimización y validación sugiere sobreajuste. Estas observaciones no autorizan modificar configuraciones después de consultar test.

In [ ]:
history_figure = plot_regression_histories(
    plot_results,
    FIGURES_DIR,
)
history_figure

## 23. Comparación de costo computacional

Una mejora pequeña puede no justificar una arquitectura mucho más lenta o grande. El tiempo depende del dispositivo y de las versiones instaladas; por tanto, sirve para comparar ejecuciones dentro de este entorno, no como benchmark universal. El número de parámetros es independiente del tiempo, pero aproxima capacidad y costo de almacenamiento.

In [ ]:
cost_table = all_results[[
    "model_name",
    "implementation_version",
    "seed",
    "n_parameters",
    "train_time_seconds",
    "inference_time_seconds",
    "best_epoch",
    "epochs_trained",
    "reached_epoch_budget",
]]
display(cost_table.sort_values("train_time_seconds"))

cost_figure = plot_regression_costs(plot_results, FIGURES_DIR)
cost_figure

## 24. Discusión teórica

| Modelo | Representación efectiva en este dataset | Sesgo principal | Riesgo interpretativo |
|---|---|---|---|
| Ridge | Vector numérico estandarizado | Aditividad lineal y contracción L2 | Confundir bajo costo con baja calidad o asumir linealidad suficiente |
| TabNet | Máscaras secuenciales sobre columnas | Selección condicional por pasos | Interpretar máscaras como causalidad |
| TabTransformer | Rama numérica con MLP; sin atención categórica activa | Composición no lineal global | Generalizar este caso degenerado a datasets categóricos |
| FT-Transformer | Un token por variable y token de agregación | Interacciones globales entre columnas | Sobreajuste y costo innecesario |
| SAINT inductivo | Tokens no lineales con atención entre columnas | Contexto intraobservación | Mayor capacidad sin aprovechar contexto entre filas |

No todas las arquitecturas deben recibir exactamente la misma representación interna: hacerlo borraría los mecanismos que se pretende estudiar. La equidad se mantiene en los datos fuente, los índices, el objetivo, el presupuesto general y la regla de selección; cada modelo conserva su tokenización coherente.

## 25. Limitaciones

1. La partición aleatoria mezcla ubicaciones cercanas y puede sobreestimar generalización geográfica por autocorrelación espacial.
2. `MedHouseVal` presenta una acumulación en el máximo observado; los residuos de alto valor están afectados por ese techo.
3. La configuración base no es una búsqueda exhaustiva de hiperparámetros.
4. Tres semillas ofrecen una estimación inicial de variabilidad, pero no sustituyen un estudio estadístico más amplio.
5. MAPE es sensible a targets pequeños y se interpreta solo junto con RMSE, MAE y R².
6. La variante inductiva de SAINT no mide el posible beneficio ni el costo metodológico de la atención entre filas.

Una extensión metodológica natural es comparar la partición aleatoria con bloques espaciales definidos antes de entrenar. Esa prueba respondería una pregunta de generalización distinta y no debe mezclarse en la misma tabla como si fuera el mismo experimento.

## 26. Conclusiones

La siguiente celda identifica el menor RMSE medio entre semillas; no lo convierte en superioridad universal. Una conclusión sólida requiere coherencia entre métricas, dispersión, diagnóstico de residuos y costo computacional.

In [ ]:
model_summary = all_results.groupby("model_name").agg(
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    mae_mean=("mae", "mean"),
    r2_mean=("r2", "mean"),
)
best_model_name = model_summary["rmse_mean"].idxmin()
best_mean = model_summary.loc[best_model_name]
observed_summary = pd.Series(
    {
        "lowest_mean_rmse_model": best_model_name,
        "n_seeds": len(EXPERIMENT_CONFIG["seeds"]),
        "rmse_mean": float(best_mean["rmse_mean"]),
        "rmse_std": float(best_mean["rmse_std"]),
        "mae_mean": float(best_mean["mae_mean"]),
        "r2_mean": float(best_mean["r2_mean"]),
        "split_fingerprint": integrity_report["shared_split_fingerprint"],
        "checkpoint_integrity": integrity_report[
            "checkpoints_present_and_reloaded"
        ],
    },
    name="observed_result",
)
display(observed_summary)

print(f"Metrics saved under: {RESULTS_DIR}")
print(f"Figures saved under: {FIGURES_DIR}")

El experimento queda trazable mediante configuraciones JSON, historiales, predicciones por fila, métricas CSV y checkpoints nativos. La tabla observada es un punto de partida empírico: cualquier afirmación posterior debe conservar el mismo aislamiento de test o declarar explícitamente un nuevo protocolo.